# YOLOv5 Traffic Sign Detection and SHAP Explainability


> **Reproducibility note:** Textual metrics and SHAP output are retained from the original course run, while selected figures are preserved under assets/. Credential uploads, archive logs, binary upload payloads, and local paths were removed. Rerunning can produce different values.


In [ ]:
from pathlib import Path
import os

SEED = 42

def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from the repository root."
    )

project_root_override = os.getenv("JTS_PROJECT_ROOT")
REPO_ROOT = (
    Path(project_root_override).expanduser().resolve()
    if project_root_override
    else find_repository_root()
)

def configured_path(variable: str, default: Path) -> Path:
    value = os.getenv(variable)
    path = Path(value).expanduser() if value else default
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()

DATASET_ROOT = configured_path("JTS_DATASET_ROOT", REPO_ROOT / "data" / "Final_Dataset")
MODELS_DIR = REPO_ROOT / "models"
ASSETS_DIR = REPO_ROOT / "assets"
MANUAL_TEST_DIR = ASSETS_DIR / "manual-tests"

def require_directory(path: Path, purpose: str) -> Path:
    if not path.is_dir():
        raise FileNotFoundError(f"{purpose} directory not found: {path}")
    return path

def require_file(path: Path, purpose: str) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"{purpose} file not found: {path}")
    return path

import random
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

random.seed(SEED)
np.random.seed(SEED)

CONFIG_TEMPLATE = REPO_ROOT / "config" / "jordanian_traffic_signs.yaml"
OUTPUTS_DIR = REPO_ROOT / "outputs" / "yolov5"
YOLOV5_DIR = configured_path("JTS_YOLOV5_DIR", REPO_ROOT / "external" / "yolov5")
YOLO_WEIGHTS = configured_path("JTS_YOLO_WEIGHTS", REPO_ROOT / "models" / "best.pt")
VIDEO_DIR = ASSETS_DIR / "demo"
YOLO_RUN_ROOT = OUTPUTS_DIR / "runs" / "train"
YOLO_RUN_NAME = "course_run"
YOLO_RUN_DIR = YOLO_RUN_ROOT / YOLO_RUN_NAME


In [ ]:
required_dataset_directories = [
    DATASET_ROOT / category / split
    for category in ("images", "labels")
    for split in ("train", "val", "test")
]
for directory in required_dataset_directories:
    require_directory(directory, "Dataset split")
print(f"Using dataset: {DATASET_ROOT}")


In [ ]:
require_file(CONFIG_TEMPLATE, "Dataset configuration template")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
runtime_config_path = OUTPUTS_DIR / "runtime_dataset.yaml"
with CONFIG_TEMPLATE.open("r", encoding="utf-8") as handle:
    runtime_config = yaml.safe_load(handle)
runtime_config["path"] = str(DATASET_ROOT)
with runtime_config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(runtime_config, handle, sort_keys=False)
print(f"Runtime YOLO dataset configuration: {runtime_config_path}")


The dataset is external. Follow `data/README.md` to download it from Kaggle.

Do not place `kaggle.json` in this repository. Use Kaggle's user-level configuration or the `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables.

Clone the YOLOv5 `v7.0` tag into `external/yolov5`, as described in the main README. The external checkout is ignored by Git.


In [ ]:
require_directory(YOLOV5_DIR, "YOLOv5 checkout")
require_file(YOLOV5_DIR / "train.py", "YOLOv5 training script")
require_file(YOLOV5_DIR / "detect.py", "YOLOv5 detection script")
print(f"Using YOLOv5 checkout: {YOLOV5_DIR}")


In [ ]:
require_directory(YOLOV5_DIR, "YOLOv5 checkout")
require_file(YOLOV5_DIR / "train.py", "YOLOv5 training script")
require_file(YOLOV5_DIR / "detect.py", "YOLOv5 detection script")
print(f"Using YOLOv5 checkout: {YOLOV5_DIR}")


In [ ]:
import torch
from IPython.display import Image, Video, clear_output

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Setup complete. Using torch {torch.__version__}")


In [ ]:
import torch
from IPython.display import Image, Video, clear_output

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Setup complete. Using torch {torch.__version__}")


In [ ]:
train_command = [
    sys.executable,
    str(YOLOV5_DIR / "train.py"),
    "--img", "416",
    "--batch", "16",
    "--epochs", "10",
    "--data", str(runtime_config_path),
    "--weights", "yolov5s.pt",
    "--cache",
    "--project", str(YOLO_RUN_ROOT),
    "--name", YOLO_RUN_NAME,
    "--exist-ok",
]
subprocess.run(train_command, cwd=YOLOV5_DIR, check=True)
YOLO_WEIGHTS = YOLO_RUN_DIR / "weights" / "best.pt"


In [ ]:
train_command = [
    sys.executable,
    str(YOLOV5_DIR / "train.py"),
    "--img", "416",
    "--batch", "16",
    "--epochs", "10",
    "--data", str(runtime_config_path),
    "--weights", "yolov5s.pt",
    "--cache",
    "--project", str(YOLO_RUN_ROOT),
    "--name", YOLO_RUN_NAME,
    "--exist-ok",
]
subprocess.run(train_command, cwd=YOLOV5_DIR, check=True)
YOLO_WEIGHTS = YOLO_RUN_DIR / "weights" / "best.pt"


In [ ]:
figure_path = require_file(YOLO_RUN_DIR / "P_curve.png", "YOLO result figure")
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()


In [ ]:
figure_path = require_file(YOLO_RUN_DIR / "R_curve.png", "YOLO result figure")
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()


In [ ]:
figure_path = require_file(YOLO_RUN_DIR / "F1_curve.png", "YOLO result figure")
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()


In [ ]:
figure_path = require_file(YOLO_RUN_DIR / "confusion_matrix.png", "YOLO result figure")
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()


train: weights=yolov5s.pt, cfg=, data=config/jordanian_traffic_signs.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=10, batch_size=16, imgsz=416, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, bucket=, cache=ram, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=runs/train, name=exp, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=-1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1, artifact_alias=latest
github: up to date with https://github.com/ultralytics/yolov5 ✅
YOLOv5 🚀 v7.0-71-gc442a2e Python-3.8.10 torch-1.13.1+cu116 CUDA:0 (Tesla T4, 15110MiB)

hyperparameters: lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005, warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1, box=0.05, cls=0.5, cls_pw=1.0, obj=1.0, obj_pw=1.0, iou_t=0.2, anchor_t=4.0, fl_gamma=0.0, hsv_h=0.015, 

In [ ]:
figure_path = require_file(YOLO_RUN_DIR / "results.png", "YOLO result figure")
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()


In [ ]:
figure_path = require_file(YOLO_RUN_DIR / "train_batch1.jpg", "YOLO result figure")
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()


<Figure size 432x288 with 1 Axes>

In [ ]:
figure_path = require_file(YOLO_RUN_DIR / "train_batch1.jpg", "YOLO result figure")
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()


<Figure size 1080x576 with 1 Axes>

In [ ]:
YOLO_WEIGHTS = require_file(YOLO_WEIGHTS, "YOLO checkpoint")

def run_detection(source: Path, name: str, image_size: int = 640, confidence: float = 0.80) -> Path:
    source = require_file(Path(source), "Detection input") if Path(source).is_file() else require_directory(Path(source), "Detection input")
    project = OUTPUTS_DIR / "detections"
    command = [
        sys.executable,
        str(YOLOV5_DIR / "detect.py"),
        "--weights", str(YOLO_WEIGHTS),
        "--img", str(image_size),
        "--conf", str(confidence),
        "--source", str(source),
        "--project", str(project),
        "--name", name,
        "--exist-ok",
    ]
    subprocess.run(command, cwd=YOLOV5_DIR, check=True)
    return project / name

test_detection_dir = run_detection(
    DATASET_ROOT / "images" / "test",
    "test_split",
    image_size=416,
    confidence=0.80,
)


<Figure size 1080x576 with 1 Axes>

In [ ]:
YOLO_WEIGHTS = require_file(YOLO_WEIGHTS, "YOLO checkpoint")

def run_detection(source: Path, name: str, image_size: int = 640, confidence: float = 0.80) -> Path:
    source = require_file(Path(source), "Detection input") if Path(source).is_file() else require_directory(Path(source), "Detection input")
    project = OUTPUTS_DIR / "detections"
    command = [
        sys.executable,
        str(YOLOV5_DIR / "detect.py"),
        "--weights", str(YOLO_WEIGHTS),
        "--img", str(image_size),
        "--conf", str(confidence),
        "--source", str(source),
        "--project", str(project),
        "--name", name,
        "--exist-ok",
    ]
    subprocess.run(command, cwd=YOLOV5_DIR, check=True)
    return project / name

test_detection_dir = run_detection(
    DATASET_ROOT / "images" / "test",
    "test_split",
    image_size=416,
    confidence=0.80,
)


<Figure size 1080x576 with 1 Axes>

In [ ]:
YOLO_WEIGHTS = require_file(YOLO_WEIGHTS, "YOLO checkpoint")
print(f"Using checkpoint: {YOLO_WEIGHTS}")


In [ ]:
manual_images = {
    "hump": require_file(MANUAL_TEST_DIR / "hump.png", "Manual test image"),
    "pedestrian_cross": require_file(MANUAL_TEST_DIR / "pedestrian_cross.png", "Manual test image"),
    "no_stop": require_file(MANUAL_TEST_DIR / "no_stop.png", "Manual test image"),
    "pass_either": require_file(MANUAL_TEST_DIR / "pass_either.png", "Manual test image"),
}


In [ ]:
manual_hump_dir = run_detection(manual_images["hump"], "manual_hump")


In [ ]:
manual_pedestrian_cross_dir = run_detection(manual_images["pedestrian_cross"], "manual_pedestrian_cross")


In [ ]:
manual_no_stop_dir = run_detection(manual_images["no_stop"], "manual_no_stop")


In [ ]:
manual_pass_either_dir = run_detection(manual_images["pass_either"], "manual_pass_either")


In [ ]:
prediction_path = require_file(manual_hump_dir / manual_images["hump"].name, "Detection result")
plt.figure(figsize=(8, 6))
plt.imshow(plt.imread(prediction_path))
plt.axis("off")
plt.show()


In [ ]:
prediction_path = require_file(manual_pedestrian_cross_dir / manual_images["pedestrian_cross"].name, "Detection result")
plt.figure(figsize=(8, 6))
plt.imshow(plt.imread(prediction_path))
plt.axis("off")
plt.show()


detect: weights=['external/yolov5/runs/train/exp/weights/best.pt'], source=external/yolov5/clas_1.png, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.8, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-71-gc442a2e Python-3.8.10 torch-1.13.1+cu116 CUDA:0 (Tesla T4, 15110MiB)

Fusing layers... 
Model summary: 157 layers, 7029004 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 external/yolov5/clas_1.png: 416x640 1 Hump, 13.8ms
Speed: 0.4ms pre-process, 13.8ms inference, 1.7ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp


In [ ]:
prediction_path = require_file(manual_no_stop_dir / manual_images["no_stop"].name, "Detection result")
plt.figure(figsize=(8, 6))
plt.imshow(plt.imread(prediction_path))
plt.axis("off")
plt.show()


detect: weights=['external/yolov5/runs/train/exp/weights/best.pt'], source=./clas_3.png, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.8, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-66-g9650f16 Python-3.8.16 torch-1.13.0+cu116 CUDA:0 (Tesla T4, 15110MiB)

Fusing layers... 
Model summary: 157 layers, 7029004 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 ./clas_3.png: 480x640 1 Pedestrian Cross, 12.0ms
Speed: 0.5ms pre-process, 12.0ms inference, 1.5ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp2


In [ ]:
prediction_path = require_file(manual_pass_either_dir / manual_images["pass_either"].name, "Detection result")
plt.figure(figsize=(8, 6))
plt.imshow(plt.imread(prediction_path))
plt.axis("off")
plt.show()


detect: weights=['external/yolov5/runs/train/exp/weights/best.pt'], source=./clas_4.png, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.8, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-66-g9650f16 Python-3.8.16 torch-1.13.0+cu116 CUDA:0 (Tesla T4, 15110MiB)

Fusing layers... 
Model summary: 157 layers, 7029004 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 ./clas_4.png: 640x640 1 No Stop, 12.6ms
Speed: 0.6ms pre-process, 12.6ms inference, 1.6ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp3


In [ ]:
prediction_path = require_file(manual_pass_either_dir / manual_images["pass_either"].name, "Detection result")
plt.figure(figsize=(8, 6))
plt.imshow(plt.imread(prediction_path))
plt.axis("off")
plt.show()


detect: weights=['external/yolov5/runs/train/exp/weights/best.pt'], source=./clas_6.png, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.8, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-66-g9650f16 Python-3.8.16 torch-1.13.0+cu116 CUDA:0 (Tesla T4, 15110MiB)

Fusing layers... 
Model summary: 157 layers, 7029004 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 ./clas_6.png: 544x640 1 Pass Either, 13.1ms
Speed: 0.6ms pre-process, 13.1ms inference, 1.5ms NMS per image at shape (1, 3, 640, 640)
Results saved to runs/detect/exp4


In [ ]:
video_source = require_file(VIDEO_DIR / "give_way_source.mp4", "Video test input")


In [ ]:
video_detection_dir = run_detection(
    video_source,
    "give_way_video",
    image_size=640,
    confidence=0.25,
)
video_prediction = require_file(
    video_detection_dir / video_source.name,
    "Video prediction",
)


In [ ]:
Video(str(video_prediction), embed=True)


In [ ]:
Video(str(video_prediction), embed=True)


Install SHAP through `requirements.txt`. The cells below explain one selected detection at a time and are computationally expensive.


In [ ]:
import json
import random

import matplotlib.patches as patches
import shap
import torch.backends.cudnn as cudnn
import torchvision
from skimage.color import rgb2gray

if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))
from utils.general import box_iou, non_max_suppression

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_half = device.type == "cuda"


In [ ]:
import json
import random

import matplotlib.patches as patches
import shap
import torch.backends.cudnn as cudnn
import torchvision
from skimage.color import rgb2gray

if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))
from utils.general import box_iou, non_max_suppression

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_half = device.type == "cuda"


In [ ]:
import json
import random

import matplotlib.patches as patches
import shap
import torch.backends.cudnn as cudnn
import torchvision
from skimage.color import rgb2gray

if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))
from utils.general import box_iou, non_max_suppression

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_half = device.type == "cuda"


In [ ]:
img_org = cv2.imread(str(manual_images["hump"]), cv2.IMREAD_COLOR)
if img_org is None:
    raise ValueError(f"OpenCV could not read {manual_images['hump']}")
img_org = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)

old_img_size = img_org.shape[:2]
ratio = float(img_size) / max(old_img_size)
new_size_y, new_size_x = (int(value * ratio) for value in old_img_size)
img = cv2.resize(img_org, (new_size_x, new_size_y))

delta_w = img_size - new_size_x
delta_h = img_size - new_size_y
top, bottom = delta_h // 2, delta_h - (delta_h // 2)
left, right = delta_w // 2, delta_w - (delta_w // 2)
img = cv2.copyMakeBorder(
    img,
    top,
    bottom,
    left,
    right,
    cv2.BORDER_CONSTANT,
    value=[0, 0, 0],
)
img = img.astype("float64") / 255
img_gray = rgb2gray(img)

plt.figure(figsize=(10, 10))
plt.imshow(img_org)
plt.axis("off")
plt.show()


In [ ]:
YOLO_WEIGHTS = require_file(YOLO_WEIGHTS, "YOLO checkpoint")
try:
    checkpoint = torch.load(
        YOLO_WEIGHTS,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(YOLO_WEIGHTS, map_location=device)
model = checkpoint["model"].float().eval().to(device)
if use_half:
    model = model.half()


In [ ]:
YOLO_WEIGHTS = require_file(YOLO_WEIGHTS, "YOLO checkpoint")
try:
    checkpoint = torch.load(
        YOLO_WEIGHTS,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(YOLO_WEIGHTS, map_location=device)
model = checkpoint["model"].float().eval().to(device)
if use_half:
    model = model.half()


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


<Figure size 1080x1080 with 1 Axes>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


torch.Size([1, 1575, 12]) # (1 image) x (1575 positions for predictions) x (4 coordinates, 1 object score, 80 class probabilities)
tensor([[ 87.12500,  65.81250, 102.12500,  78.81250,   0.83691,   1.00000]], device='cuda:0') #[x1, y1, x2, y2, class prob.*obj. score, class_index]


In [ ]:
fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(1, 1, 1)

plt.imshow(img)
for i, detection in enumerate(output[0].cpu().numpy()):
  label = f"{model.names[int(detection[5])]} {detection[4]:0.1%} ({i})"
  bbox = patches.Rectangle(detection[:2], detection[2]-detection[0], detection[3]-detection[1], linewidth=3, edgecolor='g', facecolor='none')
  plt.text(detection[0], detection[1], label, color="red")
  ax.add_patch(bbox)
plt.show()


<Figure size 1080x1080 with 1 Axes>

In [ ]:
class CastNumpy(torch.nn.Module):
  def __init__(self):
    super(CastNumpy, self).__init__()
  
  def forward(self, image):
    """
    In the forward function we accept the inputs and cast them to a pytorch tensor
    """
    
    image = np.ascontiguousarray(image)
    image = torch.from_numpy(image).to(device)
    if image.ndimension() == 3:
      image = image.unsqueeze(0)
    return image.half() if use_half else image.float()

numpy2torch_converter = CastNumpy()


In [ ]:
numpy2torch_converter.forward(img.transpose(2, 0, 1))


tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]], device='cuda:0', dtype=torch.float16)

In [ ]:
class OD2Score(torch.nn.Module):
    def __init__(self, target, conf_thresh=0.01, iou_thresh=0.5):
        """
        constructor
        """
        super(OD2Score, self).__init__()
        self.conf_thresh = conf_thresh
        self.iou_thresh = iou_thresh
        self.target = torch.tensor(target, device=device)

    def forward(self, x):
        """
        In the forward function we accept the predictions and return the score for a selected target of the box
        """
        
        score_best_box = torch.zeros([x[0].shape[0]], device=device)

        for idx, img in enumerate(x[0]):
          img = img.unsqueeze(0)
        
          output = non_max_suppression(img, conf_thres=self.conf_thresh, iou_thres=self.iou_thresh)
          if output and output[0] is not None:
            correct_class_predictions = output[0][...,5] == self.target[5]
            correctly_labeled_boxes = output[0][correct_class_predictions]
            
            if correctly_labeled_boxes.shape[0]:
              iou_with_target, _idx = box_iou(correctly_labeled_boxes[:,:4], self.target.unsqueeze(0)[:,:4]).max(1)
              index_best_box_in_correct_class = torch.argmax(iou_with_target)
              index_best_box_in_output = torch.where(output[0][...,5] == self.target[5])[0][index_best_box_in_correct_class]

              # score for the detection is the score from the prediction steps times intersection with expected position: (object score x class score) x iou_with_target

              # classification score
              #score_best_box[idx] = output[0][index_best_box_in_output][4]
              # position score
              #score_best_box[idx] = iou_with_target[index_best_box_in_correct_class]
              # both
              score_best_box[idx] = output[0][index_best_box_in_output][4]*iou_with_target[index_best_box_in_correct_class]

        return score_best_box.detach().cpu().numpy()

# look at the image output above and set the index of the object you want to analyze here
target_index = 0
target = output[0].cpu().numpy()[target_index,:]
scoring = OD2Score(target, conf_thresh=confidence, iou_thresh=iou)


In [26]:
class SuperPixler(torch.nn.Module):
  def __init__(self, image, super_pixel_width):
    super(SuperPixler, self).__init__()
    
    self.image = image.transpose(2, 0, 1) # model expects images in BRG, not RGB, so transpose color channels
    self.mean_color = self.image.mean()
    self.image = np.expand_dims(self.image, axis=0)
    self.image_width = image.shape[1]
    self.super_pixel_width = super_pixel_width

  
  def forward(self, x):
    """
    In the forward step we accept the super pixel masks and transform them to a batch of images
    """

    pixeled_image = np.repeat(self.image.copy(), x.shape[0], axis=0)

    for i, super_pixel in enumerate(x.T):
      images_to_pixelate = [bool(p) for p in super_pixel]
      x = (i*self.super_pixel_width//self.image_width)*self.super_pixel_width
      y = i*self.super_pixel_width%self.image_width
      pixeled_image[images_to_pixelate,:,x:x+self.super_pixel_width, y:y+self.super_pixel_width] = self.mean_color
    
    return pixeled_image

super_pixel_width = 16
assert (img_size/super_pixel_width)%1 == 0, "image width needs to be multiple of super pixel width"   
n_super_pixel = int((img.shape[1]/super_pixel_width)**2)

super_pixler = SuperPixler(img, super_pixel_width = super_pixel_width)

In [ ]:
img_org = cv2.imread(str(manual_images["pedestrian_cross"]), cv2.IMREAD_COLOR)
if img_org is None:
    raise ValueError(f"OpenCV could not read {manual_images['pedestrian_cross']}")
img_org = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)

old_img_size = img_org.shape[:2]
ratio = float(img_size) / max(old_img_size)
new_size_y, new_size_x = (int(value * ratio) for value in old_img_size)
img = cv2.resize(img_org, (new_size_x, new_size_y))

delta_w = img_size - new_size_x
delta_h = img_size - new_size_y
top, bottom = delta_h // 2, delta_h - (delta_h // 2)
left, right = delta_w // 2, delta_w - (delta_w // 2)
img = cv2.copyMakeBorder(
    img,
    top,
    bottom,
    left,
    right,
    cv2.BORDER_CONSTANT,
    value=[0, 0, 0],
)
img = img.astype("float64") / 255
img_gray = rgb2gray(img)

plt.figure(figsize=(10, 10))
plt.imshow(img_org)
plt.axis("off")
plt.show()


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


0.00%


  0%|          | 0/1 [00:00<?, ?it/s]

4.00%


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

...

The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

100 non-zero shap values found


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


<IPython.core.display.HTML object>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


<Figure size 1080x1080 with 1 Axes>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


<Figure size 1080x1080 with 1 Axes>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


torch.Size([1, 1575, 12]) # (1 image) x (1575 positions for predictions) x (4 coordinates, 1 object score, 80 class probabilities)
tensor([[21.48438, 60.21875, 33.37500, 72.50000,  0.67139,  3.00000]], device='cuda:0') #[x1, y1, x2, y2, class prob.*obj. score, class_index]


In [ ]:
fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(1, 1, 1)

plt.imshow(img)
for i, detection in enumerate(output[0].cpu().numpy()):
  label = f"{model.names[int(detection[5])]} {detection[4]:0.1%} ({i})"
  bbox = patches.Rectangle(detection[:2], detection[2]-detection[0], detection[3]-detection[1], linewidth=3, edgecolor='g', facecolor='none')
  plt.text(detection[0], detection[1], label, color="red")
  ax.add_patch(bbox)
plt.show()


<Figure size 1080x1080 with 1 Axes>

In [ ]:
class CastNumpy(torch.nn.Module):
  def __init__(self):
    super(CastNumpy, self).__init__()
  
  def forward(self, image):
    """
    In the forward function we accept the inputs and cast them to a pytorch tensor
    """
    
    image = np.ascontiguousarray(image)
    image = torch.from_numpy(image).to(device)
    if image.ndimension() == 3:
      image = image.unsqueeze(0)
    return image.half() if use_half else image.float()

numpy2torch_converter = CastNumpy()


In [ ]:
numpy2torch_converter.forward(img.transpose(2, 0, 1))


tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]], device='cuda:0', dtype=torch.float16)

In [ ]:
class OD2Score(torch.nn.Module):
    def __init__(self, target, conf_thresh=0.01, iou_thresh=0.5):
        """
        constructor
        """
        super(OD2Score, self).__init__()
        self.conf_thresh = conf_thresh
        self.iou_thresh = iou_thresh
        self.target = torch.tensor(target, device=device)

    def forward(self, x):
        """
        In the forward function we accept the predictions and return the score for a selected target of the box
        """
        
        score_best_box = torch.zeros([x[0].shape[0]], device=device)

        for idx, img in enumerate(x[0]):
          img = img.unsqueeze(0)
        
          output = non_max_suppression(img, conf_thres=self.conf_thresh, iou_thres=self.iou_thresh)
          if output and output[0] is not None:
            correct_class_predictions = output[0][...,5] == self.target[5]
            correctly_labeled_boxes = output[0][correct_class_predictions]
            
            if correctly_labeled_boxes.shape[0]:
              iou_with_target, _idx = box_iou(correctly_labeled_boxes[:,:4], self.target.unsqueeze(0)[:,:4]).max(1)
              index_best_box_in_correct_class = torch.argmax(iou_with_target)
              index_best_box_in_output = torch.where(output[0][...,5] == self.target[5])[0][index_best_box_in_correct_class]

              # score for the detection is the score from the prediction steps times intersection with expected position: (object score x class score) x iou_with_target

              # classification score
              #score_best_box[idx] = output[0][index_best_box_in_output][4]
              # position score
              #score_best_box[idx] = iou_with_target[index_best_box_in_correct_class]
              # both
              score_best_box[idx] = output[0][index_best_box_in_output][4]*iou_with_target[index_best_box_in_correct_class]

        return score_best_box.detach().cpu().numpy()

# look at the image output above and set the index of the object you want to analyze here
target_index = 0
target = output[0].cpu().numpy()[target_index,:]
scoring = OD2Score(target, conf_thresh=confidence, iou_thresh=iou)


In [39]:
class SuperPixler(torch.nn.Module):
  def __init__(self, image, super_pixel_width):
    super(SuperPixler, self).__init__()
    
    self.image = image.transpose(2, 0, 1) # model expects images in BRG, not RGB, so transpose color channels
    self.mean_color = self.image.mean()
    self.image = np.expand_dims(self.image, axis=0)
    self.image_width = image.shape[1]
    self.super_pixel_width = super_pixel_width

  
  def forward(self, x):
    """
    In the forward step we accept the super pixel masks and transform them to a batch of images
    """

    pixeled_image = np.repeat(self.image.copy(), x.shape[0], axis=0)

    for i, super_pixel in enumerate(x.T):
      images_to_pixelate = [bool(p) for p in super_pixel]
      x = (i*self.super_pixel_width//self.image_width)*self.super_pixel_width
      y = i*self.super_pixel_width%self.image_width
      pixeled_image[images_to_pixelate,:,x:x+self.super_pixel_width, y:y+self.super_pixel_width] = self.mean_color
    
    return pixeled_image

super_pixel_width = 16
assert (img_size/super_pixel_width)%1 == 0, "image width needs to be multiple of super pixel width"   
n_super_pixel = int((img.shape[1]/super_pixel_width)**2)

super_pixler = SuperPixler(img, super_pixel_width = super_pixel_width)

In [ ]:
img_org = cv2.imread(str(manual_images["no_stop"]), cv2.IMREAD_COLOR)
if img_org is None:
    raise ValueError(f"OpenCV could not read {manual_images['no_stop']}")
img_org = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)

old_img_size = img_org.shape[:2]
ratio = float(img_size) / max(old_img_size)
new_size_y, new_size_x = (int(value * ratio) for value in old_img_size)
img = cv2.resize(img_org, (new_size_x, new_size_y))

delta_w = img_size - new_size_x
delta_h = img_size - new_size_y
top, bottom = delta_h // 2, delta_h - (delta_h // 2)
left, right = delta_w // 2, delta_w - (delta_w // 2)
img = cv2.copyMakeBorder(
    img,
    top,
    bottom,
    left,
    right,
    cv2.BORDER_CONSTANT,
    value=[0, 0, 0],
)
img = img.astype("float64") / 255
img_gray = rgb2gray(img)

plt.figure(figsize=(10, 10))
plt.imshow(img_org)
plt.axis("off")
plt.show()


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


0.00%


  0%|          | 0/1 [00:00<?, ?it/s]

4.00%


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

...

The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

100 non-zero shap values found


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


<IPython.core.display.HTML object>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


<Figure size 1080x1080 with 1 Axes>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


<Figure size 1080x1080 with 1 Axes>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


torch.Size([1, 1575, 12]) # (1 image) x (1575 positions for predictions) x (4 coordinates, 1 object score, 80 class probabilities)
tensor([[ 83.50000,  56.75000, 100.37500,  78.62500,   0.84717,   4.00000]], device='cuda:0') #[x1, y1, x2, y2, class prob.*obj. score, class_index]


In [ ]:
fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(1, 1, 1)

plt.imshow(img)
for i, detection in enumerate(output[0].cpu().numpy()):
  label = f"{model.names[int(detection[5])]} {detection[4]:0.1%} ({i})"
  bbox = patches.Rectangle(detection[:2], detection[2]-detection[0], detection[3]-detection[1], linewidth=3, edgecolor='g', facecolor='none')
  plt.text(detection[0], detection[1], label, color="red")
  ax.add_patch(bbox)
plt.show()


<Figure size 1080x1080 with 1 Axes>

In [ ]:
class CastNumpy(torch.nn.Module):
  def __init__(self):
    super(CastNumpy, self).__init__()
  
  def forward(self, image):
    """
    In the forward function we accept the inputs and cast them to a pytorch tensor
    """
    
    image = np.ascontiguousarray(image)
    image = torch.from_numpy(image).to(device)
    if image.ndimension() == 3:
      image = image.unsqueeze(0)
    return image.half() if use_half else image.float()

numpy2torch_converter = CastNumpy()


In [ ]:
numpy2torch_converter.forward(img.transpose(2, 0, 1))


tensor([[[[0.00000, 0.00000, 0.40381,  ..., 0.28638, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.30200,  ..., 0.25879, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.41187,  ..., 0.25488, 0.00000, 0.00000],
          ...,
          [0.00000, 0.00000, 0.64697,  ..., 0.74121, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.64697,  ..., 0.66260, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.65869,  ..., 0.62744, 0.00000, 0.00000]],

         [[0.00000, 0.00000, 0.36084,  ..., 0.25879, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.29028,  ..., 0.23535, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.43921,  ..., 0.23132, 0.00000, 0.00000],
          ...,
          [0.00000, 0.00000, 0.64697,  ..., 0.73730, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.64697,  ..., 0.65869, 0.00000, 0.00000],
          [0.00000, 0.00000, 0.65869,  ..., 0.61963, 0.00000, 0.00000]],

         [[0.00000, 0.00000, 0.20789,  ..., 0.14905, 0.00000, 0.00000],
          [0.00000, 0.00000, 0

In [ ]:
class OD2Score(torch.nn.Module):
    def __init__(self, target, conf_thresh=0.01, iou_thresh=0.5):
        """
        constructor
        """
        super(OD2Score, self).__init__()
        self.conf_thresh = conf_thresh
        self.iou_thresh = iou_thresh
        self.target = torch.tensor(target, device=device)

    def forward(self, x):
        """
        In the forward function we accept the predictions and return the score for a selected target of the box
        """
        
        score_best_box = torch.zeros([x[0].shape[0]], device=device)

        for idx, img in enumerate(x[0]):
          img = img.unsqueeze(0)
        
          output = non_max_suppression(img, conf_thres=self.conf_thresh, iou_thres=self.iou_thresh)
          if output and output[0] is not None:
            correct_class_predictions = output[0][...,5] == self.target[5]
            correctly_labeled_boxes = output[0][correct_class_predictions]
            
            if correctly_labeled_boxes.shape[0]:
              iou_with_target, _idx = box_iou(correctly_labeled_boxes[:,:4], self.target.unsqueeze(0)[:,:4]).max(1)
              index_best_box_in_correct_class = torch.argmax(iou_with_target)
              index_best_box_in_output = torch.where(output[0][...,5] == self.target[5])[0][index_best_box_in_correct_class]

              # score for the detection is the score from the prediction steps times intersection with expected position: (object score x class score) x iou_with_target

              # classification score
              #score_best_box[idx] = output[0][index_best_box_in_output][4]
              # position score
              #score_best_box[idx] = iou_with_target[index_best_box_in_correct_class]
              # both
              score_best_box[idx] = output[0][index_best_box_in_output][4]*iou_with_target[index_best_box_in_correct_class]

        return score_best_box.detach().cpu().numpy()

# look at the image output above and set the index of the object you want to analyze here
target_index = 0
target = output[0].cpu().numpy()[target_index,:]
scoring = OD2Score(target, conf_thresh=confidence, iou_thresh=iou)


In [50]:
class SuperPixler(torch.nn.Module):
  def __init__(self, image, super_pixel_width):
    super(SuperPixler, self).__init__()
    
    self.image = image.transpose(2, 0, 1) # model expects images in BRG, not RGB, so transpose color channels
    self.mean_color = self.image.mean()
    self.image = np.expand_dims(self.image, axis=0)
    self.image_width = image.shape[1]
    self.super_pixel_width = super_pixel_width

  
  def forward(self, x):
    """
    In the forward step we accept the super pixel masks and transform them to a batch of images
    """

    pixeled_image = np.repeat(self.image.copy(), x.shape[0], axis=0)

    for i, super_pixel in enumerate(x.T):
      images_to_pixelate = [bool(p) for p in super_pixel]
      x = (i*self.super_pixel_width//self.image_width)*self.super_pixel_width
      y = i*self.super_pixel_width%self.image_width
      pixeled_image[images_to_pixelate,:,x:x+self.super_pixel_width, y:y+self.super_pixel_width] = self.mean_color
    
    return pixeled_image

super_pixel_width = 16
assert (img_size/super_pixel_width)%1 == 0, "image width needs to be multiple of super pixel width"   
n_super_pixel = int((img.shape[1]/super_pixel_width)**2)

super_pixler = SuperPixler(img, super_pixel_width = super_pixel_width)

In [ ]:
img_org = cv2.imread(str(manual_images["pass_either"]), cv2.IMREAD_COLOR)
if img_org is None:
    raise ValueError(f"OpenCV could not read {manual_images['pass_either']}")
img_org = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)

old_img_size = img_org.shape[:2]
ratio = float(img_size) / max(old_img_size)
new_size_y, new_size_x = (int(value * ratio) for value in old_img_size)
img = cv2.resize(img_org, (new_size_x, new_size_y))

delta_w = img_size - new_size_x
delta_h = img_size - new_size_y
top, bottom = delta_h // 2, delta_h - (delta_h // 2)
left, right = delta_w // 2, delta_w - (delta_w // 2)
img = cv2.copyMakeBorder(
    img,
    top,
    bottom,
    left,
    right,
    cv2.BORDER_CONSTANT,
    value=[0, 0, 0],
)
img = img.astype("float64") / 255
img_gray = rgb2gray(img)

plt.figure(figsize=(10, 10))
plt.imshow(img_org)
plt.axis("off")
plt.show()


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


0.00%


  0%|          | 0/1 [00:00<?, ?it/s]

4.00%


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

...

The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

93 non-zero shap values found


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


<IPython.core.display.HTML object>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


<Figure size 1080x1080 with 1 Axes>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


<Figure size 1080x1080 with 1 Axes>

In [ ]:
torch_image = torch.from_numpy(
    np.ascontiguousarray(img.transpose(2, 0, 1))
).to(device).unsqueeze(0).float()
if use_half:
    torch_image = torch_image.half()

prediction = model(torch_image)
print(
    prediction[0].shape,
    "# batch x positions x (4 coordinates, 1 object score, 7 class probabilities)",
)
output = non_max_suppression(
    prediction[0],
    conf_thres=confidence,
    iou_thres=iou,
)
print(output[0], "#[x1, y1, x2, y2, confidence, class_index]")


torch.Size([1, 1575, 12]) # (1 image) x (1575 positions for predictions) x (4 coordinates, 1 object score, 80 class probabilities)
tensor([[ 56.25000,  46.62500,  82.62500, 107.00000,   0.90723,   6.00000]], device='cuda:0') #[x1, y1, x2, y2, class prob.*obj. score, class_index]


In [ ]:
fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(1, 1, 1)

plt.imshow(img)
for i, detection in enumerate(output[0].cpu().numpy()):
  label = f"{model.names[int(detection[5])]} {detection[4]:0.1%} ({i})"
  bbox = patches.Rectangle(detection[:2], detection[2]-detection[0], detection[3]-detection[1], linewidth=3, edgecolor='g', facecolor='none')
  plt.text(detection[0], detection[1], label, color="red")
  ax.add_patch(bbox)
plt.show()


<Figure size 1080x1080 with 1 Axes>

In [ ]:
class CastNumpy(torch.nn.Module):
  def __init__(self):
    super(CastNumpy, self).__init__()
  
  def forward(self, image):
    """
    In the forward function we accept the inputs and cast them to a pytorch tensor
    """
    
    image = np.ascontiguousarray(image)
    image = torch.from_numpy(image).to(device)
    if image.ndimension() == 3:
      image = image.unsqueeze(0)
    return image.half() if use_half else image.float()

numpy2torch_converter = CastNumpy()


In [ ]:
numpy2torch_converter.forward(img.transpose(2, 0, 1))


tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]], device='cuda:0', dtype=torch.float16)

In [ ]:
class OD2Score(torch.nn.Module):
    def __init__(self, target, conf_thresh=0.01, iou_thresh=0.5):
        """
        constructor
        """
        super(OD2Score, self).__init__()
        self.conf_thresh = conf_thresh
        self.iou_thresh = iou_thresh
        self.target = torch.tensor(target, device=device)

    def forward(self, x):
        """
        In the forward function we accept the predictions and return the score for a selected target of the box
        """
        
        score_best_box = torch.zeros([x[0].shape[0]], device=device)

        for idx, img in enumerate(x[0]):
          img = img.unsqueeze(0)
        
          output = non_max_suppression(img, conf_thres=self.conf_thresh, iou_thres=self.iou_thresh)
          if output and output[0] is not None:
            correct_class_predictions = output[0][...,5] == self.target[5]
            correctly_labeled_boxes = output[0][correct_class_predictions]
            
            if correctly_labeled_boxes.shape[0]:
              iou_with_target, _idx = box_iou(correctly_labeled_boxes[:,:4], self.target.unsqueeze(0)[:,:4]).max(1)
              index_best_box_in_correct_class = torch.argmax(iou_with_target)
              index_best_box_in_output = torch.where(output[0][...,5] == self.target[5])[0][index_best_box_in_correct_class]

              # score for the detection is the score from the prediction steps times intersection with expected position: (object score x class score) x iou_with_target

              # classification score
              #score_best_box[idx] = output[0][index_best_box_in_output][4]
              # position score
              #score_best_box[idx] = iou_with_target[index_best_box_in_correct_class]
              # both
              score_best_box[idx] = output[0][index_best_box_in_output][4]*iou_with_target[index_best_box_in_correct_class]

        return score_best_box.detach().cpu().numpy()

# look at the image output above and set the index of the object you want to analyze here
target_index = 0
target = output[0].cpu().numpy()[target_index,:]
scoring = OD2Score(target, conf_thresh=confidence, iou_thresh=iou)


In [61]:
class SuperPixler(torch.nn.Module):
  def __init__(self, image, super_pixel_width):
    super(SuperPixler, self).__init__()
    
    self.image = image.transpose(2, 0, 1) # model expects images in BRG, not RGB, so transpose color channels
    self.mean_color = self.image.mean()
    self.image = np.expand_dims(self.image, axis=0)
    self.image_width = image.shape[1]
    self.super_pixel_width = super_pixel_width

  
  def forward(self, x):
    """
    In the forward step we accept the super pixel masks and transform them to a batch of images
    """

    pixeled_image = np.repeat(self.image.copy(), x.shape[0], axis=0)

    for i, super_pixel in enumerate(x.T):
      images_to_pixelate = [bool(p) for p in super_pixel]
      x = (i*self.super_pixel_width//self.image_width)*self.super_pixel_width
      y = i*self.super_pixel_width%self.image_width
      pixeled_image[images_to_pixelate,:,x:x+self.super_pixel_width, y:y+self.super_pixel_width] = self.mean_color
    
    return pixeled_image

super_pixel_width = 16
assert (img_size/super_pixel_width)%1 == 0, "image width needs to be multiple of super pixel width"   
n_super_pixel = int((img.shape[1]/super_pixel_width)**2)

super_pixler = SuperPixler(img, super_pixel_width = super_pixel_width)

In [62]:
super_pixel_model = torch.nn.Sequential(
    super_pixler,
    numpy2torch_converter,
    model,
    scoring
)

In [63]:
# use Kernel SHAP to explain the detection

background_super_pixel = np.array([[1 for _ in range(n_super_pixel)]])
image_super_pixel = np.array([[0 for _ in range(n_super_pixel)]])
kernel_explainer = shap.KernelExplainer(super_pixel_model, background_super_pixel)

# Very large values for nsamples cause OOM errors depending on image and super pixel parameter. We combine batches of SHAP values to distribute the load. 
collected_shap_values = np.zeros_like(background_super_pixel)
b = 25
for i in range(b):
  print(f"{i/b:0.2%}")

  shap_values = kernel_explainer.shap_values(image_super_pixel, nsamples=1000)

  # take shap value with highest abs. value for each pixel from each batch
  stacked_values = np.vstack([shap_values, collected_shap_values])
  index_max_values = np.argmax(np.abs(stacked_values), axis=0)
  collected_shap_values = stacked_values[index_max_values, range(shap_values.shape[1])]
print((collected_shap_values!=0).sum(), "non-zero shap values found")

# plot the found SHAP values. Expected value does not match due to merging of batches
shap.initjs()
shap.force_plot(kernel_explainer.expected_value, collected_shap_values)

0.00%


  0%|          | 0/1 [00:00<?, ?it/s]

4.00%


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

...

The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


  0%|          | 0/1 [00:00<?, ?it/s]

99 non-zero shap values found


The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 


<IPython.core.display.HTML object>

In [64]:
# match super pixels back to image pixels
shap_to_pixel = collected_shap_values.reshape(img_size//super_pixel_width, img_size//super_pixel_width)# reshape to square
shap_to_pixel = np.repeat(shap_to_pixel, super_pixel_width, axis=0)# extend superpixles to the right
shap_to_pixel = np.repeat(shap_to_pixel, super_pixel_width, axis=1)# and down
shap_to_pixel = shap_to_pixel/(np.max(np.abs(collected_shap_values))*2) + 0.5 # center values between 0 and 1 for the colour map

In [65]:
# plot image and shap values for super pixels on top
fig, ax = plt.subplots(1, 1, figsize=(15,15))
ax.set_title("Super pixel contribution to target detection")
ax.imshow(img_gray, alpha=0.7, cmap='gray', vmin=0, vmax=1)
ax.imshow(shap_to_pixel, cmap=plt.cm.seismic, vmin=0, vmax=1, alpha=0.5)

# Add bounding box of target
label = f"{model.names[int(target[5])]} {target[4]:0.1%}"
plt.text(target[0], target[1], label, color="green")
bbox = patches.Rectangle(target[:2], target[2]-target[0], target[3]-target[1], linewidth=1, edgecolor='g', facecolor='none')
ax.add_patch(bbox)
plt.show()

<Figure size 1080x1080 with 1 Axes>